# STAT 4830 — Token-Level Slop Classifier (v1.5)

This notebook is an **ablation between v1 and v2**. It keeps v1's causal
transformer backbone (`distilgpt2`) but replaces v1's corruption-based slop
generator with v2's mirror-prompted slop. Everything else matches v1 as
closely as possible.

## Why this notebook exists

v1 used `distilgpt2` + corruption-based slop and reported very high accuracy.
v2 swapped two things at once: bidirectional backbone (`distilbert-base-uncased`)
**and** mirror-prompted slop. That makes it impossible to tell which change
caused which effect on the numbers. v1.5 isolates the slop-generator change so
the comparison is clean:

| Notebook | Backbone | Slop generator | Isolates |
|---|---|---|---|
| v1 | distilgpt2 (causal) | corruption | original baseline |
| **v1.5** | **distilgpt2 (causal)** | **mirror prompting (Qwen 0.5B)** | **slop quality effect** |
| v2 | distilbert-base-uncased (bidirectional) | mirror prompting (Qwen 0.5B) | slop + backbone effects |

The interesting comparisons are:
- **v1 vs v1.5** isolates the effect of replacing corruption with real LLM slop,
  with the backbone held constant. If v1 reports near-100% and v1.5 drops, that
  is direct evidence that v1's accuracy was inflated by corruption artifacts.
- **v1.5 vs v2** isolates the effect of switching from a causal to a bidirectional
  backbone, with the slop generator held constant. If v2 outperforms v1.5, that
  is direct evidence that bidirectional context per token actually matters for
  slop classification.

## What this notebook does (unchanged from v1's structure)

1. Loads human essays from `essays.csv`
2. **Generates mirror-prompted slop with Qwen 0.5B** (the only substantive change)
3. Trains a frozen-`distilgpt2` mean-pool logistic regression baseline
4. Trains a token-level slop classifier on top of `distilgpt2`'s final hidden states
5. Aggregates token slop probabilities into a document-level slop score with
   mean pooling, and *additionally* reports bottom-k pooling at eval time
6. Visualizes which tokens look most slop-like

## Mathematical formulation

For tokens $x_1, \ldots, x_T$ with attention mask $m_1, \ldots, m_T$:

$$h_t = \mathrm{distilgpt2}(x)_t \quad\text{(final hidden state per token, causal)}$$
$$p_t = \sigma(w^\top h_t + b) \quad\text{(per-token P(human))}$$

**Mean pooling** (training + one eval):
$$\hat{y}_{\text{mean}}(x) = \frac{\sum_{t=1}^T m_t p_t}{\sum_{t=1}^T m_t}$$

**Bottom-k pooling** (second eval only, same trained model):
$$\hat{y}_{\text{bot-}k}(x) = \frac{1}{k}\sum_{t \in \mathrm{bot}_k(p)} p_t$$

We minimize document-level binary cross-entropy on $\hat{y}_{\text{mean}}$:
$$\min_\theta \sum_i \mathrm{BCE}(\hat{y}_{\text{mean}}(x^{(i)}), y^{(i)})$$

where $y^{(i)} = 1$ for human and $y^{(i)} = 0$ for slop.

## Note on causal context

`distilgpt2` is a causal transformer: each token's final hidden state only
attends to itself and prior tokens. That means the per-token slop probability
$p_t$ is computed without knowledge of tokens at positions $t+1, \ldots, T$.
For *generation* this is the correct constraint; for *classification* it is
a handicap. We expect this to matter most for the token-level heatmap and
slightly less for the aggregated document score (because mean pooling
averages over all positions including the final token, whose embedding *does*
see the full document).


## 0) Runtime

Colab GPU (T4 or better) strongly recommended. The mirror-slop generation step
uses a small 0.5B-parameter model — fast on a T4, painful on CPU. All generated
mirror slop is cached to `mirror_slop_cache.csv` so you only pay generation
cost once.

If you already ran v2 in the same Colab session, the mirror cache may already
exist and this notebook will reuse it (the cache is generator-output, not
classifier-specific).


In [ ]:
# Colab: uncomment
!pip -q install transformers datasets accelerate scikit-learn pandas matplotlib tqdm


In [ ]:
import os
import re
import json
import random
import gc
import traceback
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1) Configuration

Note that `MODEL_NAME` is **`distilgpt2`** (matching v1), not `distilbert-base-uncased`.
Everything else matches v2 so the comparison is apples-to-apples.


In [ ]:
# =========================
# Configuration
# =========================

DATA_PATH = "essays.csv"
TEXT_COL = "essay"

MAX_HUMAN_SAMPLES = 1200
MIN_CHAR_LEN = 500
MAX_CHAR_LEN = 12000

# --- Classifier backbone (UNCHANGED FROM v1: causal transformer) ---
MODEL_NAME = "distilgpt2"

# --- Slop generator (CHANGED FROM v1: real LLM mirror prompting, matches v2) ---
SLOP_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MIRROR_CACHE_PATH = "mirror_slop_cache.csv"
N_MIRROR_SAMPLES = 1200         # cap; auto-clamps to len(df_human). Each ~2-4s on T4.
GEN_MAX_NEW_TOKENS = 400
GEN_TEMPERATURE = 0.8
GEN_TOP_P = 0.95
MIN_MIRROR_WORDS = 20
MIN_MIRROR_CHARS = 120

# Tokenization / training
MAX_LEN = 256
TRAIN_BS = 16
EVAL_BS = 32
LR = 2e-5
EPOCHS = 3
WEIGHT_DECAY = 0.01
DROPOUT = 0.1
GRAD_CLIP = 1.0
EMBED_BATCH_SIZE = 32

# Bottom-k pooling ablation (eval-time only)
BOTTOM_K = 20

assert os.path.exists(DATA_PATH), f"Could not find DATA_PATH={DATA_PATH}"
print("DATA_PATH:", DATA_PATH)
print("Classifier backbone:", MODEL_NAME, "(causal — matches v1)")
print("Slop generator:", SLOP_MODEL_NAME, "(mirror prompting — matches v2)")


## 2) Load human essays

Identical to v1 and v2.


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print("raw shape:", df_raw.shape)
print("columns:", list(df_raw.columns))

assert TEXT_COL in df_raw.columns, f"Expected a '{TEXT_COL}' column in essays.csv"

df_human = df_raw[[TEXT_COL]].copy()
df_human = df_human.rename(columns={TEXT_COL: "text"})
df_human["text"] = df_human["text"].astype(str).str.strip()

df_human = df_human[df_human["text"].str.len().between(MIN_CHAR_LEN, MAX_CHAR_LEN)]
df_human = df_human.drop_duplicates(subset=["text"]).reset_index(drop=True)

if len(df_human) > MAX_HUMAN_SAMPLES:
    df_human = df_human.sample(MAX_HUMAN_SAMPLES, random_state=SEED).reset_index(drop=True)

df_human["label"] = 1   # 1 = human
print("filtered human shape:", df_human.shape)
df_human.head()


## 3) Mirror-prompted slop generation

This is the part that changed from v1. Instead of corrupting human text by
shuffling sentences and injecting filler phrases, we generate **real LLM slop**
by asking a small instruction model to continue each human essay in matching
style and length.

### The prompt pattern (Pangram Section 4.2)

For each human essay:
1. Extract the first 1-2 sentences as a "seed."
2. Count the words in the original essay as a target length.
3. Ask the slop model to continue the essay in matching style.

This forces the AI side of the dataset to share topic, length, and opening
with its human counterpart, so the classifier cannot exploit those features
as shortcuts. It is forced to learn *stylistic* LLM tells.

### Why a small local model?

`Qwen2.5-0.5B-Instruct` is ~1 GB, runs on a Colab T4, and is a coherent
instruction-follower. We are not trying to match GPT-4 here — we just need
the slop to look like *real LLM writing* rather than *corrupted human writing*.
A small model's natural tells (mild repetition, hedging, formulaic transitions)
are representative of the slop we want to detect.

### Caching

Generated mirrors are written to `mirror_slop_cache.csv`. Reruns reuse the
cache. If v2 already populated this cache in the same Colab session, this
notebook will reuse it directly — the slop is generator output, not
classifier-specific.


In [ ]:
# Seed sentences from each human essay
def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if s.strip()]

def get_seed(text, n_seed_sents=2, max_seed_chars=400):
    sents = split_sentences(text)
    if len(sents) >= n_seed_sents:
        seed = " ".join(sents[:n_seed_sents])
    else:
        seed = text[:max_seed_chars]
    if len(seed) > max_seed_chars:
        seed = seed[:max_seed_chars]
    return seed

def build_mirror_prompt(human_text):
    seed = get_seed(human_text)
    target_words = max(120, min(400, len(human_text.split())))
    return (
        f"Continue writing the following essay. Match its topic and style, "
        f"and write approximately {target_words} additional words. "
        f"Do not include a title, preamble, or meta-commentary — "
        f"just continue the essay directly.\n\nEssay so far:\n{seed}"
    )

# Strip the most common AI boilerplate tells (Pangram Section 3.10.2)
BOILERPLATE_PREFIXES = [
    "sure", "here is", "here's", "certainly", "of course", "title:",
    "abstract:", "i'd be happy", "i am happy", "i'm happy", "i have:",
    "continuation:", "continued:", "essay:",
]

def strip_boilerplate(gen_text):
    t = gen_text.strip()
    paras = t.split("\n\n", 1)
    first = paras[0].strip().lower()
    if any(first.startswith(p) for p in BOILERPLATE_PREFIXES):
        t = paras[1] if len(paras) > 1 else ""
    return t.strip()


In [ ]:
# ========================================================================
# Slop generator: cache validation + load
# ========================================================================
NEEDED = min(N_MIRROR_SAMPLES, len(df_human))

need_generate = True
if os.path.exists(MIRROR_CACHE_PATH):
    try:
        cached = pd.read_csv(MIRROR_CACHE_PATH)
    except Exception:
        cached = pd.DataFrame()
    if len(cached) >= NEEDED and len(cached) > 0:
        print(f"Using cached mirror slop: {len(cached)} rows")
        need_generate = False
    else:
        print(f"Found stale/empty cache ({len(cached)} rows, need {NEEDED}). Removing.")
        os.remove(MIRROR_CACHE_PATH)

if torch.cuda.is_available():
    print("CUDA OK:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — generation will be slow. "
          "In Colab: Runtime → Change runtime type → T4 GPU.")

# Clear stale model handles before reloading
for _name in ("slop_model", "slop_tokenizer"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if need_generate:
    print(f"Loading slop generator: {SLOP_MODEL_NAME}")
    slop_tokenizer = AutoTokenizer.from_pretrained(SLOP_MODEL_NAME)
    slop_model = AutoModelForCausalLM.from_pretrained(
        SLOP_MODEL_NAME,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    ).to(device)
    slop_model.eval()
    print("Slop generator loaded. param device:", next(slop_model.parameters()).device)
else:
    slop_tokenizer = None
    slop_model = None


In [ ]:
# ========================================================================
# Single-example generation helper + sanity check
# ========================================================================

@torch.no_grad()
def generate_one_mirror(human_text):
    prompt = build_mirror_prompt(human_text)
    messages = [{"role": "user", "content": prompt}]
    prompt_text = slop_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    enc = slop_tokenizer(prompt_text, return_tensors="pt").to(device)
    out = slop_model.generate(
        **enc,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=True,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        pad_token_id=slop_tokenizer.eos_token_id,
    )
    gen = slop_tokenizer.decode(
        out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True
    )
    return strip_boilerplate(gen)

if need_generate:
    print("\n--- Sanity check: one generation ---")
    _sanity = generate_one_mirror(df_human.iloc[0]["text"])
    print(f"OK: produced {len(_sanity.split())} words, {len(_sanity)} chars")
    print("first 300 chars:", repr(_sanity[:300]))
    assert len(_sanity.split()) >= 10, (
        "Sanity generation produced <10 words. STOP and investigate."
    )


In [ ]:
# ========================================================================
# Full generation loop with REAL error visibility
# ========================================================================

if need_generate:
    n_gen = min(N_MIRROR_SAMPLES, len(df_human))
    mirror_texts, source_texts = [], []
    n_errors, n_too_short = 0, 0

    for i in tqdm(range(n_gen), desc="Generating mirror slop"):
        src = df_human.iloc[i]["text"]
        try:
            mirror = generate_one_mirror(src)
        except Exception as e:
            n_errors += 1
            if n_errors <= 3:
                print(f"\n[{i}] error type: {type(e).__name__}")
                traceback.print_exc()
            continue
        if (len(mirror.split()) < MIN_MIRROR_WORDS or
            len(mirror) < MIN_MIRROR_CHARS):
            n_too_short += 1
            if n_too_short <= 3:
                print(f"\n[{i}] too short: {len(mirror.split())} words, "
                      f"{len(mirror)} chars; first 200: {mirror[:200]!r}")
            continue
        mirror_texts.append(mirror)
        source_texts.append(src)

    print(f"\nkept: {len(mirror_texts)}  errors: {n_errors}  too_short: {n_too_short}")

    if len(mirror_texts) == 0:
        raise RuntimeError("No usable mirrors produced. See errors above.")

    cache_df = pd.DataFrame({
        "source_human": source_texts,
        "mirror_slop": mirror_texts,
    })
    cache_df.to_csv(MIRROR_CACHE_PATH, index=False)
    print(f"Saved {len(cache_df)} mirror examples to {MIRROR_CACHE_PATH}")

# Free the slop model now that generation is done
if "slop_model" in globals() and slop_model is not None:
    del slop_model
if "slop_tokenizer" in globals() and slop_tokenizer is not None:
    del slop_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cache_df = pd.read_csv(MIRROR_CACHE_PATH)
print("mirror cache shape:", cache_df.shape)
assert len(cache_df) > 0, "cache is empty — cannot proceed"


In [ ]:
# Build the slop dataframe. Label 0 = slop, label 1 = human.
df_slop = pd.DataFrame({"text": cache_df["mirror_slop"].astype(str).str.strip()})
df_slop = df_slop[df_slop["text"].str.len() >= MIN_MIRROR_CHARS].reset_index(drop=True)
df_slop["label"] = 0

# Keep only the human essays that actually have a mirror in the cache
df_human_matched = df_human.iloc[:len(df_slop)].reset_index(drop=True)

print("human:", df_human_matched.shape, "slop:", df_slop.shape)
print("\n--- Example HUMAN (first 400 chars) ---")
print(df_human_matched.loc[0, "text"][:400])
print("\n--- Example MIRROR SLOP (first 400 chars) ---")
print(df_slop.loc[0, "text"][:400])


## 4) Train / val / test split

In [ ]:
df_all = pd.concat([df_human_matched, df_slop], axis=0, ignore_index=True)
df_all = df_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df_all, test_size=0.30, random_state=SEED, stratify=df_all["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, d.shape, d["label"].value_counts().to_dict())


## 5) Tokenizer + backbone (causal — matches v1)

We use **distilgpt2**, a causal transformer. GPT-style tokenizers do not
define a pad token by default, so we set `pad_token = eos_token` (this is
the same workaround v1 used).

Important caveat: because the backbone is causal, each token's final hidden
state $h_t$ only attends to positions $1, \ldots, t$. The slop probability
$p_t = \sigma(w^\top h_t + b)$ at position $t$ is therefore computed without
knowledge of any tokens *after* $t$. Bidirectional models (BERT-family) do
not have this restriction. v2 isolates exactly this difference.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

backbone_for_baseline = AutoModel.from_pretrained(MODEL_NAME).to(device)
backbone_for_baseline.eval()

hidden_size = backbone_for_baseline.config.hidden_size
print("hidden_size:", hidden_size)
print("pad_token_id:", tokenizer.pad_token_id)


## 6) Baseline — frozen distilgpt2 + logistic regression

Same baseline as v1: pass each text through frozen distilgpt2, mean-pool the
final hidden states across tokens, and fit a logistic regression on top.
This tells us how well the *pretrained* causal-transformer features separate
human from mirror slop without any task-specific training.


In [ ]:
def embed_texts_mean_pool(texts, tokenizer, backbone, max_len=256, batch_size=32):
    all_vecs = []
    backbone.eval()
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(
            batch_texts, truncation=True, padding=True,
            max_length=max_len, return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = backbone(**enc, return_dict=True)
            hidden = out.last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            all_vecs.append(pooled.detach().cpu().numpy())
    return np.concatenate(all_vecs, axis=0)

X_train = embed_texts_mean_pool(train_df["text"].tolist(), tokenizer, backbone_for_baseline, max_len=MAX_LEN, batch_size=EMBED_BATCH_SIZE)
X_val = embed_texts_mean_pool(val_df["text"].tolist(), tokenizer, backbone_for_baseline, max_len=MAX_LEN, batch_size=EMBED_BATCH_SIZE)
X_test = embed_texts_mean_pool(test_df["text"].tolist(), tokenizer, backbone_for_baseline, max_len=MAX_LEN, batch_size=EMBED_BATCH_SIZE)

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

logreg = LogisticRegression(max_iter=2000, random_state=SEED)
logreg.fit(X_train, y_train)

val_probs = logreg.predict_proba(X_val)[:, 1]
test_probs = logreg.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

baseline_metrics = {
    "val_auc": roc_auc_score(y_val, val_probs),
    "test_acc": accuracy_score(y_test, test_preds),
    "test_f1": f1_score(y_test, test_preds),
    "test_auc": roc_auc_score(y_test, test_probs),
}
print("=== Frozen distilgpt2 + logistic regression baseline ===")
print(json.dumps(baseline_metrics, indent=2))
print("Confusion matrix:\n", confusion_matrix(y_test, test_preds))

# free baseline backbone memory
del backbone_for_baseline
if device.type == "cuda":
    torch.cuda.empty_cache()


## 7) Dataset class for the token-level classifier

In [ ]:
class SlopDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }

train_ds = SlopDataset(train_df, tokenizer, max_len=MAX_LEN)
val_ds = SlopDataset(val_df, tokenizer, max_len=MAX_LEN)
test_ds = SlopDataset(test_df, tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=TRAIN_BS, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=EVAL_BS, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=EVAL_BS, shuffle=False)

len(train_ds), len(val_ds), len(test_ds)


## 8) Token-level slop classifier (causal backbone, matches v1)

Architecture (identical to v1):
1. Text → distilgpt2 → per-token final hidden states (causal)
2. Linear head per token → token logit → sigmoid → per-token P(human)
3. Masked mean over tokens → document P(human)
4. BCE against the document label

The forward pass returns the raw per-token probabilities so we can apply
**different pooling strategies at eval time** (mean vs bottom-k) without
retraining.


In [ ]:
class GPTTokenSlopClassifier(nn.Module):
    def __init__(self, model_name="distilgpt2", dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.token_classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        hidden = out.last_hidden_state
        hidden = self.dropout(hidden)

        token_logits = self.token_classifier(hidden).squeeze(-1)     # [B, T]
        token_probs = torch.sigmoid(token_logits)                    # [B, T]

        mask = attention_mask.float()
        doc_score_mean = (token_probs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        loss = None
        if labels is not None:
            loss = nn.BCELoss()(doc_score_mean, labels.float())

        return {
            "loss": loss,
            "token_probs": token_probs,
            "doc_score_mean": doc_score_mean,
            "mask": mask,
        }

model = GPTTokenSlopClassifier(MODEL_NAME, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

num_training_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * num_training_steps)),
    num_training_steps=num_training_steps,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")


## 9) Bottom-k pooling helper

After the forward pass we have per-token P(human) for every example. Bottom-k
pooling takes the **k lowest** token probabilities — the k most slop-like
tokens — and averages them. The intuition: if even the most slop-looking
region of the document still looks human, the whole document is probably human.

We use bottom-k rather than top-k because label=1 is human (high = human),
so the least-human tokens are the ones we care about for detecting slop.

**Caveat for the causal backbone.** Because distilgpt2 is causal, the per-token
score $p_t$ is computed without knowledge of tokens after position $t$. The
"most slop-like token" identified by bottom-k pooling will systematically
under-use information available later in the document. We expect this to hurt
bottom-k pooling more than mean pooling for the causal backbone, and v2's
bidirectional version to outperform v1.5 most visibly on this metric.


In [ ]:
def bottom_k_pool(token_probs, mask, k=20):
    '''
    token_probs: [B, T]  per-token P(human)
    mask:        [B, T]  1 for real tokens, 0 for padding
    Returns:     [B]     mean of the k smallest masked token probs per example
    '''
    masked_probs = token_probs.masked_fill(mask == 0, float("inf"))
    B, T = masked_probs.shape
    k_eff = min(k, T)
    bot_vals, _ = torch.topk(-masked_probs, k=k_eff, dim=1)
    bot_vals = -bot_vals
    finite_mask = torch.isfinite(bot_vals).float()
    bot_vals = torch.where(torch.isfinite(bot_vals), bot_vals, torch.zeros_like(bot_vals))
    doc = (bot_vals * finite_mask).sum(dim=1) / finite_mask.sum(dim=1).clamp(min=1.0)
    return doc


## 10) Training loop

Trained with BCE on the mean-pooled document score. Best checkpoint selected
by validation AUC.


In [ ]:
def run_eval(model, loader, k=BOTTOM_K):
    model.eval()
    all_probs_mean, all_probs_botk, all_labels, all_loss = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            all_loss.append(out["loss"].item())

            probs_mean = out["doc_score_mean"]
            probs_botk = bottom_k_pool(out["token_probs"], out["mask"], k=k)

            all_probs_mean.extend(probs_mean.detach().cpu().numpy().tolist())
            all_probs_botk.extend(probs_botk.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

    all_probs_mean = np.array(all_probs_mean)
    all_probs_botk = np.array(all_probs_botk)
    all_labels = np.array(all_labels)

    def metrics_for(probs, tag):
        preds = (probs >= 0.5).astype(int)
        return {
            f"{tag}_acc": accuracy_score(all_labels, preds),
            f"{tag}_f1": f1_score(all_labels, preds),
            f"{tag}_auc": roc_auc_score(all_labels, probs),
        }

    out_metrics = {"loss": float(np.mean(all_loss))}
    out_metrics.update(metrics_for(all_probs_mean, "mean"))
    out_metrics.update(metrics_for(all_probs_botk, "botk"))
    return out_metrics, all_labels, all_probs_mean, all_probs_botk

best_val_auc = -1
best_state = None
history = []

for epoch in range(EPOCHS):
    model.train()
    train_losses = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out["loss"]

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())

    val_metrics, _, _, _ = run_eval(model, val_loader)
    row = {
        "epoch": epoch + 1,
        "train_loss": float(np.mean(train_losses)),
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)
    print(row)

    if val_metrics["mean_auc"] > best_val_auc:
        best_val_auc = val_metrics["mean_auc"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

hist_df = pd.DataFrame(history)
hist_df


## 11) Final evaluation — mean vs bottom-k pooling

Same trained model, two pooling strategies at test time. Compare these numbers
side-by-side with the v2 numbers (same slop, same training recipe, different
backbone) — that comparison isolates the effect of bidirectional vs causal
context per token.


In [ ]:
model.load_state_dict(best_state)
model.to(device)

test_metrics, test_labels, test_probs_mean, test_probs_botk = run_eval(model, test_loader)

print("=== Test metrics (v1.5: distilgpt2 + mirror slop) ===")
print(json.dumps(test_metrics, indent=2))

print("\n--- Mean pooling ---")
preds_mean = (test_probs_mean >= 0.5).astype(int)
print("Confusion matrix:\n", confusion_matrix(test_labels, preds_mean))
print(classification_report(test_labels, preds_mean, digits=4))

print("\n--- Bottom-k pooling (k=%d) ---" % BOTTOM_K)
preds_botk = (test_probs_botk >= 0.5).astype(int)
print("Confusion matrix:\n", confusion_matrix(test_labels, preds_botk))
print(classification_report(test_labels, preds_botk, digits=4))

# ROC curves
fpr_m, tpr_m, _ = roc_curve(test_labels, test_probs_mean)
fpr_b, tpr_b, _ = roc_curve(test_labels, test_probs_botk)

plt.figure(figsize=(6, 5))
plt.plot(fpr_m, tpr_m, label=f"mean pool (AUC={test_metrics['mean_auc']:.4f})")
plt.plot(fpr_b, tpr_b, label=f"bot-{BOTTOM_K} pool (AUC={test_metrics['botk_auc']:.4f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC — v1.5 (distilgpt2 + mirror slop)")
plt.legend()
plt.show()

# Score distributions
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, probs, title in [
    (axes[0], test_probs_mean, "Mean pooling"),
    (axes[1], test_probs_botk, f"Bottom-{BOTTOM_K} pooling"),
]:
    ax.hist(probs[test_labels == 1], bins=25, alpha=0.6, label="human")
    ax.hist(probs[test_labels == 0], bins=25, alpha=0.6, label="slop")
    ax.set_xlabel("Predicted P(human)")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()


## 12) Token heatmap

Per-token P(human) rendered as a red-shaded heatmap. Reminder: with a causal
backbone, the score at position $t$ does not see tokens at positions
$> t$. Compare this heatmap visually to v2's heatmap on the same example —
v2's tokens should look more uniformly informed because each token's score
benefits from full bidirectional context.


In [ ]:
def score_single_text(model, text, tokenizer, max_len=256):
    model.eval()
    enc = tokenizer(
        text, truncation=True, padding="max_length",
        max_length=max_len, return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attention_mask)

    token_probs = out["token_probs"][0].detach().cpu().numpy()
    doc_score_mean = float(out["doc_score_mean"][0].detach().cpu().item())
    doc_score_botk = float(
        bottom_k_pool(out["token_probs"], out["mask"], k=BOTTOM_K)[0].detach().cpu().item()
    )

    input_ids_cpu = enc["input_ids"][0].tolist()
    attn_cpu = enc["attention_mask"][0].tolist()
    toks = tokenizer.convert_ids_to_tokens(input_ids_cpu)

    pairs = []
    for tok, p, m in zip(toks, token_probs, attn_cpu):
        if m == 0:
            continue
        # GPT-2 BPE uses Ġ to mark word-initial tokens
        clean = tok.replace("Ġ", " ")
        pairs.append((clean, float(p)))
    return {
        "doc_score_mean": doc_score_mean,
        "doc_score_botk": doc_score_botk,
        "token_pairs": pairs,
    }

def render_token_heatmap(token_pairs):
    '''Red intensity grows as P(human) drops, i.e. as the token looks more slop-like.'''
    parts = []
    for tok, p in token_pairs:
        slop_intensity = 1.0 - p
        alpha = min(0.90, max(0.05, slop_intensity))
        parts.append(
            f'<span style="background-color: rgba(255,0,0,{alpha:.2f}); '
            f'padding:2px; margin:1px; border-radius:3px;">{tok}</span>'
        )
    return "".join(parts)

from IPython.display import HTML, display

example_text = test_df.iloc[0]["text"]
example_label = test_df.iloc[0]["label"]
scored = score_single_text(model, example_text, tokenizer, max_len=MAX_LEN)

print("true label (1=human, 0=slop):", example_label)
print("doc score (mean):", round(scored["doc_score_mean"], 4))
print(f"doc score (bot-{BOTTOM_K}):", round(scored["doc_score_botk"], 4))
display(HTML(render_token_heatmap(scored["token_pairs"])))


## 13) Qualitative error analysis

Most "human-like" slop and most "slop-like" human, ranked by mean-pool score.
This is the error surface that hard negative mining (Pangram Algorithm 1)
would feed back into a retraining round.


In [ ]:
def predict_dataframe_scores(model, df, loader):
    model.eval()
    mean_scores, botk_scores = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            mean_scores.extend(out["doc_score_mean"].detach().cpu().numpy().tolist())
            botk_scores.extend(
                bottom_k_pool(out["token_probs"], out["mask"], k=BOTTOM_K)
                .detach().cpu().numpy().tolist()
            )
    scored = df.copy().reset_index(drop=True)
    scored["p_human_mean"] = mean_scores
    scored["p_human_botk"] = botk_scores
    scored["pred_mean"] = (scored["p_human_mean"] >= 0.5).astype(int)
    return scored

test_scored = predict_dataframe_scores(model, test_df, test_loader)

pd.set_option("display.max_colwidth", 200)

print("Most 'human-like' slop (true label=0, highest P(human) under mean pooling):")
display(
    test_scored[test_scored["label"] == 0]
    .sort_values("p_human_mean", ascending=False)
    [["p_human_mean", "p_human_botk", "pred_mean", "text"]]
    .head(5)
)

print("\nMost 'slop-like' human (true label=1, lowest P(human) under mean pooling):")
display(
    test_scored[test_scored["label"] == 1]
    .sort_values("p_human_mean", ascending=True)
    [["p_human_mean", "p_human_botk", "pred_mean", "text"]]
    .head(5)
)


## 14) Save model + tokenizer

In [ ]:
SAVE_DIR = "./token_slop_classifier_v15"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model.pt"))
tokenizer.save_pretrained(SAVE_DIR)

with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump({
        "notebook_version": "v1.5",
        "model_name": MODEL_NAME,
        "slop_model_name": SLOP_MODEL_NAME,
        "max_len": MAX_LEN,
        "dropout": DROPOUT,
        "bottom_k": BOTTOM_K,
        "baseline_metrics": baseline_metrics,
        "test_metrics": test_metrics,
    }, f, indent=2)

print("saved to:", SAVE_DIR)


## 15) Reading the v1 ↔ v1.5 ↔ v2 comparison

The point of running v1.5 is the **clean two-step ablation**:

| Step | Notebooks | What changes | What it isolates |
|---|---|---|---|
| A | v1 → v1.5 | corruption slop → mirror slop (backbone unchanged) | how much of v1's accuracy was real vs artifact |
| B | v1.5 → v2 | causal → bidirectional backbone (slop unchanged) | how much bidirectional context per token actually helps |

### What to expect

**v1 → v1.5 (slop swap).** v1's reported accuracy is almost certainly inflated
because the corruption signature (filler prefixes, repeated n-grams) is trivial
to detect. Mirror-prompted slop strips that crutch away. v1.5's accuracy
should be lower than v1's, and that drop is the *honest* baseline — it tells
you what the classifier can actually do when the slop looks like real LLM
writing instead of corrupted human writing. If v1.5 is dramatically lower
than v1, then v1's high numbers were measuring corruption detection, not
LLM-style detection.

**v1.5 → v2 (backbone swap).** v2's backbone (DistilBERT) is bidirectional, so
each per-token slop probability sees the entire document instead of only the
left context. We expect v2 to outperform v1.5, especially on:
- the bottom-k pooling metric, because the "most slop-like token" estimate is
  more accurate when each token has full context
- the token heatmap interpretability, because per-token scores are not biased
  toward the left side of the document

If v2 does *not* outperform v1.5, that is itself a meaningful finding — it
would suggest that for *this* slop generator and *this* dataset size, the
final-token embedding of a causal model already captures enough document-level
context to do most of the work, and the marginal value of bidirectional
context per token is small.

### What this tells us about Pangram

Pangram's deep-classifier-vs-probability-feature argument (their Section 3.4)
is not really about causal vs bidirectional — it is about *learned representations
of LLM voice* vs *probability statistics from a smaller proxy LM*. Both v1.5 and
v2 use learned representations, so both are on the right side of that argument.
The v1.5 ↔ v2 comparison is a more granular question: *given that you use
learned representations, does bidirectional encoding matter?*


## 16) Summary

### What this notebook does
- Causal `distilgpt2` backbone (matches v1)
- Mirror-prompted slop from Qwen 0.5B (matches v2)
- Frozen-backbone logistic regression baseline
- Token-level classifier head with mean-pool BCE training
- Mean *and* bottom-k pooling at eval time
- Token heatmap and qualitative error analysis

### What it does not do
- Hard negative mining (Pangram Algorithm 1) — see v2's section 15 stub
- Slop generator training — separate project track
- LoRA fine-tuning of the backbone — scaling concern, deferred

### Numbers to report alongside v1 and v2

| Condition | Test Acc | Test AUC | Notes |
|---|---|---|---|
| v1: distilgpt2 + corruption | (from v1) | (from v1) | inflated by corruption signature |
| **v1.5: distilgpt2 + mirror** | **(this nb)** | **(this nb)** | causal backbone, real slop |
| v2: DistilBERT + mirror | (from v2) | (from v2) | bidirectional backbone, real slop |

Pair the two ablations:
- **v1 vs v1.5** = the slop-quality story (was v1's accuracy real?)
- **v1.5 vs v2** = the backbone story (does bidirectional help?)
